# 04 — Library card exploration

Clustering by card type, discipline requirement, cost, and text features.
Cluster profiling with radar charts, and Gradient Boosting to surface
attributes associated with competitive play.

**Discipline features — two sources:**
1. `library_disc_df` — from the structured `discipline` column (full
   names: "Auspex", "Animalism & Auspex"...). Always present.
2. `library_tag_df` — from bracket tags **at the start of a line** in
   `card_text` (e.g. `[dom]` / `[DOM]`). These mark alternate-discipline
   versions of the card, with the same uppercase = superior convention as
   crypt. Mid-line tags (e.g. "React with Conviction": requires [chi]...)
   are excluded by the regex.

**Prerequisite:** run `01_data.ipynb` first.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import json

import numpy as np
import pandas as pd
import plotly  # Fixing dynamic exports
import plotly.express as px
import plotly.graph_objects as go
from helpers import (
    DISC_CODE_NAME,
    SEED,
    artifact_path,
    fit_supervised,
    fit_transform,
    normalize_rows,
    tolist,
)
from plotly.subplots import make_subplots
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    silhouette_score,  # pyright: ignore[reportUnknownVariableType]
)
from sklearn.model_selection import train_test_split  # pyright: ignore[reportUnknownVariableType]

# Reproductibility
np.random.seed(SEED)

# Dynamic graphs export
plotly.io.renderers.default = "notebook"  # pyright: ignore[reportUnknownMemberType]
plotly.offline.init_notebook_mode()

## Load artifacts

In [ ]:
df_library = pd.read_parquet(artifact_path("df_library.parquet"))
library_disc_df = pd.read_parquet(artifact_path("library_disc_df.parquet"))
library_tag_df = pd.read_parquet(artifact_path("library_tag_df.parquet"))
library_type_df = pd.read_parquet(artifact_path("library_type_df.parquet"))
X_library_scaled = np.load(artifact_path("X_library_scaled.npy"))
with open(artifact_path("X_library_cols.json"), encoding="utf-8") as f:
    X_library_cols: list[str] = json.load(f)

type_cols = library_type_df.columns.tolist()
disc_name_cols = library_disc_df.columns.tolist()
tag_cols_base = [c for c in library_tag_df.columns if c.islower()]

n_with_tags = (library_tag_df > 0).any(axis=1).sum()
print(f"{len(df_library)} library cards, {X_library_scaled.shape[1]} features")
print(f"Discipline columns (field): {disc_name_cols}")
print(f"Tag columns (line-start): {library_tag_df.shape[1]}, covering {n_with_tags} cards")

## SVD reduction

In [ ]:
svd_lib = TruncatedSVD(n_components=50, random_state=SEED)
X_library_reduced = fit_transform(svd_lib, X_library_scaled)
print(
    f"TruncatedSVD 50 components -> "
    f"{svd_lib.explained_variance_ratio_.sum():.1%} variance explained"
)

svd_2d = TruncatedSVD(n_components=2, random_state=SEED)
X_library_2d = svd_2d.fit_transform(X_library_scaled)

## K-Means -- elbow + silhouette

In [ ]:
k_range = range(2, 21)
inertias: list[float] = []
silhouettes: list[float] = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_library_reduced)
    inertias.append(km.inertia_)
    silhouettes.append(float(silhouette_score(X_library_reduced, labels)))

fig = make_subplots(rows=1, cols=2, subplot_titles=["Elbow (inertia)", "Silhouette score"])
fig.add_trace(
    go.Scatter(x=list(k_range), y=inertias, mode="lines+markers", name="Inertia"), row=1, col=1
)
fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=silhouettes,
        mode="lines+markers",
        name="Silhouette",
        line=dict(color="darkorange"),
    ),
    row=1,
    col=2,
)
fig.update_xaxes(title_text="k")
fig.update_layout(title="Library card clustering -- choose k", showlegend=False)
fig.show()

In [ ]:
silhouettes_smooth: np.ndarray = (
    pd.Series(silhouettes).rolling(window=3, center=True, min_periods=1).mean().to_numpy()
)
k_library = max(4, int(np.argmax(silhouettes_smooth)) + 2)
print(f"k selected: {k_library}")

In [ ]:
km_library = KMeans(n_clusters=k_library, random_state=SEED, n_init=10)
df_library["cluster"] = km_library.fit_predict(X_library_reduced).astype(str)

fig = px.scatter(  # pyright: ignore[reportUnknownMemberType]
    x=X_library_2d[:, 0],
    y=X_library_2d[:, 1],
    color=df_library["cluster"],
    hover_name=df_library["name"],
    hover_data={
        "type": df_library["type"],
        "era": df_library["era"],
        "play_rate": df_library["play_rate"].round(3),
    },
    labels={
        "x": f"SVD1 ({svd_2d.explained_variance_ratio_[0]:.1%})",
        "y": f"SVD2 ({svd_2d.explained_variance_ratio_[1]:.1%})",
        "color": "Cluster",
    },
    title=f"Library cards -- K-Means k={k_library}, SVD 2D",
    category_orders={"color": [str(i) for i in range(k_library)]},
)
fig.show()

## Cluster profiling

In [ ]:
# --- Card type radar per cluster ---
type_matrix = library_type_df.groupby(df_library["cluster"]).mean()
spoke_types = [c.removeprefix("type_") for c in type_cols]

# --- Pool cost vs play rate scatter ---
cost_by_cluster: pd.Series = (
    df_library["pool_cost_num"]
    .where(df_library["pool_cost_num"] >= 0)  # exclude X=-1
    .groupby(df_library["cluster"])
    .mean()
    .sort_index()
)
play_by_cluster: pd.Series = df_library.groupby("cluster")["play_rate"].mean().sort_index()

fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "polar"}, {"type": "xy"}]],
    subplot_titles=[
        "Card type share per cluster",
        "Avg pool cost vs avg play rate",
    ],
    column_widths=[0.65, 0.35],
)

for cl in sorted(type_matrix.index):
    vals = tolist(type_matrix.loc[cl])
    fig.add_trace(
        go.Scatterpolar(  # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
            r=vals + [vals[0]],
            theta=spoke_types + [spoke_types[0]],
            name=f"Cluster {cl}",
            fill="toself",
            opacity=0.6,
        ),
        row=1,
        col=1,
    )

clusters_sorted: list[str] = sorted(play_by_cluster.index)
fig.add_trace(
    go.Scatter(
        x=[float(play_by_cluster[cl]) for cl in clusters_sorted],
        y=[float(cost_by_cluster.get(cl, float("nan"))) for cl in clusters_sorted],
        mode="markers+text",
        text=[f"Cluster {cl}" for cl in clusters_sorted],
        textposition="top center",
        showlegend=False,
        marker=dict(size=8, color="steelblue"),
    ),
    row=1,
    col=2,
)
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, float(type_matrix.values.max())])),
    xaxis=dict(title="Avg play rate"),
    yaxis=dict(title="Avg pool cost"),
    title="Card type profile per cluster",
)
fig.show()

In [ ]:
# --- Discipline radar (from `discipline` field, full names) ---
disc_name_matrix = library_disc_df.groupby(df_library["cluster"]).mean()


# Keep only disciplines that appear in at least one cluster with mean > 0.01
active_discs = disc_name_matrix.columns[disc_name_matrix.max() > 0.01].tolist()
disc_name_top = disc_name_matrix[active_discs] if active_discs else disc_name_matrix

fig = go.Figure()
for cl in sorted(disc_name_top.index):
    vals = tolist(disc_name_top.loc[cl])
    fig.add_trace(
        go.Scatterpolar(  # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
            r=vals + [vals[0]],
            theta=active_discs + [active_discs[0]],
            name=f"Cluster {cl}",
            fill="toself",
            opacity=0.6,
        )
    )
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, float(disc_name_top.values.max())])),
    title="Discipline requirement per cluster (from `discipline` field)",
)
fig.show()

In [ ]:
# --- Line-start bracket tag radar (base level only) ---
if tag_cols_base:
    tag_matrix = pd.DataFrame(
        {
            cl: library_tag_df.loc[df_library["cluster"] == cl, tag_cols_base].mean()
            for cl in sorted(df_library["cluster"].unique())
        }
    ).T

    active_tags = tag_matrix.columns[tag_matrix.max() > 0.01].tolist()
    tag_labels = [DISC_CODE_NAME.get(t, t) for t in active_tags]

    fig = go.Figure()
    for cl in sorted(tag_matrix.index):
        vals = tolist(tag_matrix.loc[cl, active_tags])
        fig.add_trace(
            go.Scatterpolar(  # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
                r=vals + [vals[0]],
                theta=tag_labels + [tag_labels[0]],
                name=f"Cluster {cl}",
                fill="toself",
                opacity=0.6,
            )
        )
    fig.update_layout(
        polar=dict(
            radialaxis=dict(visible=True, range=[0, float(tag_matrix[active_tags].values.max())])
        ),
        title="Alternate-discipline tags per cluster (line-start [xxx] in card_text)",
    )
    fig.show()
else:
    print("No line-start discipline tags found in this dataset.")

In [ ]:
# --- Era distribution per cluster (stacked bar) ---
era_counts = normalize_rows(df_library.groupby(["cluster", "era"]).size().unstack(fill_value=0))
fig = go.Figure()
for era in era_counts.columns:
    fig.add_trace(
        go.Bar(
            name=era,
            x=[f"Cluster {cl}" for cl in sorted(era_counts.index)],
            y=[float(era_counts.loc[cl, era]) for cl in sorted(era_counts.index)],
        )
    )
fig.update_layout(
    barmode="stack",
    title="Era distribution per cluster (share of cards)",
    xaxis_title="Cluster",
    yaxis_title="Share",
)
fig.show()

## Why are some library cards played and others not?

In [ ]:
modeling_df = df_library[~df_library["banned"]].copy()
X_model = X_library_scaled[~df_library["banned"].values]
y_model = modeling_df["played"].astype(int)

print(
    f"Class balance: {y_model.mean():.1%} played "
    f"({int(y_model.sum())} / {len(y_model)} non-banned cards)"
)

min_class = y_model.value_counts().min() if y_model.nunique() >= 2 else 0
if min_class < 2:
    print("Cannot fit classifier: a class has fewer than 2 members.")
    print(y_model.value_counts())
else:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_model, y_model, test_size=0.25, random_state=SEED, stratify=y_model
    )
    gbc = GradientBoostingClassifier(random_state=SEED)
    fit_supervised(gbc, X_tr, y_tr)
    auc = roc_auc_score(y_te, gbc.predict_proba(X_te)[:, 1])
    print(f"Held-out ROC-AUC: {auc:.3f}")

    top = (
        pd.Series(gbc.feature_importances_, index=X_library_cols)
        .sort_values(ascending=False)
        .head(15)
        .sort_values()
    )
    top_df = pd.DataFrame({"feature": top.index, "importance": top.values})
    fig = px.bar(  # pyright: ignore[reportUnknownMemberType]
        top_df,
        x="importance",
        y="feature",
        orientation="h",
        title="Attributes associated with a library card being played (GB importance)",
        labels={"importance": "Importance", "feature": ""},
    )
    fig.update_layout(yaxis=dict(tickfont=dict(size=11)))
    fig.show()

In [ ]:
print("Mean play rate by era (non-banned):")
print(modeling_df.groupby("era")["play_rate"].mean().sort_values(ascending=False).round(3))

print("\nMean play rate by card type (exploded):")
exploded = modeling_df[["play_rate", "type"]].copy()
exploded["type_list"] = exploded["type"].fillna("").str.split("/")
exploded = exploded.explode("type_list")
exploded["type_list"] = exploded["type_list"].str.strip()
print(exploded.groupby("type_list")["play_rate"].mean().sort_values(ascending=False).round(3))